# Phase 0: Data Exploration — ColliderML pileup_only_pu0

Goals:
1. Load pileup_only_pu0 subsets (particles, tracker_hits, calo_hits)
2. Characterise single min-bias interactions: particle multiplicities, pT spectra, hit counts
3. Understand detector geometry: layers, cells, subsystem codes
4. Build the particle → hits mapping
5. Visualise example events in (η, φ) and (x, y, z)

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from tqdm.auto import tqdm

from genpu.data import (
    load_from_hf,
    explode_list_columns,
    get_num_events,
    get_particle_hit_map,
    compute_eta_phi,
)

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12})

## 1. Load Data

In [ ]:
# Load the three key tables — this downloads from HuggingFace on first run
print("Loading particles...")
particles_table = load_from_hf("particles")
print(f"  {get_num_events(particles_table)} events, columns: {particles_table.column_names}")

print("Loading tracker hits...")
tracker_table = load_from_hf("tracker_hits")
print(f"  {get_num_events(tracker_table)} events, columns: {tracker_table.column_names}")

print("Loading calo hits...")
calo_table = load_from_hf("calo_hits")
print(f"  {get_num_events(calo_table)} events, columns: {calo_table.column_names}")

## 2. Particle Multiplicity & Kinematics

In [ ]:
n_events = get_num_events(particles_table)
n_sample = min(n_events, 5000)  # use up to 5k events for distributions

multiplicities = []
all_pt = []
all_eta = []
all_pdg = []
primary_fracs = []

for i in tqdm(range(n_sample), desc="Scanning particles"):
    ev = explode_list_columns(particles_table, i)
    n_part = len(ev['particle_id'])
    multiplicities.append(n_part)
    
    pt, eta, phi = compute_eta_phi(ev['px'], ev['py'], ev['pz'])
    all_pt.append(pt)
    all_eta.append(eta)
    all_pdg.append(ev['pdg_id'])
    
    if 'primary' in ev:
        primary_fracs.append(ev['primary'].sum() / max(n_part, 1))

multiplicities = np.array(multiplicities)
all_pt_flat = np.concatenate(all_pt)
all_eta_flat = np.concatenate(all_eta)
all_pdg_flat = np.concatenate(all_pdg)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Multiplicity distribution
axes[0, 0].hist(multiplicities, bins=80, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Particle multiplicity per event')
axes[0, 0].set_ylabel('Events')
axes[0, 0].set_title(f'Particle multiplicity (mean={multiplicities.mean():.1f}, std={multiplicities.std():.1f})')

# pT spectrum (log scale)
axes[0, 1].hist(all_pt_flat, bins=np.logspace(-3, 2, 100), edgecolor='black', alpha=0.7)
axes[0, 1].set_xscale('log')
axes[0, 1].set_yscale('log')
axes[0, 1].set_xlabel('pT [GeV/c]')
axes[0, 1].set_ylabel('Particles')
axes[0, 1].set_title('Transverse momentum spectrum')

# Eta distribution
axes[1, 0].hist(all_eta_flat[np.isfinite(all_eta_flat)], bins=100, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('η (pseudorapidity)')
axes[1, 0].set_ylabel('Particles')
axes[1, 0].set_title('Pseudorapidity distribution')

# PDG ID histogram (top species)
from collections import Counter
pdg_counts = Counter(all_pdg_flat.astype(int))
top_pdg = pdg_counts.most_common(15)
pdg_labels = [str(p) for p, _ in top_pdg]
pdg_vals = [c for _, c in top_pdg]
axes[1, 1].barh(pdg_labels, pdg_vals, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Count')
axes[1, 1].set_title('Top 15 particle species (PDG ID)')

plt.tight_layout()
plt.savefig('../plots/phase0_particle_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSummary statistics:")
print(f"  Events sampled: {n_sample}")
print(f"  Multiplicity: mean={multiplicities.mean():.1f}, median={np.median(multiplicities):.0f}, min={multiplicities.min()}, max={multiplicities.max()}")
print(f"  pT: median={np.median(all_pt_flat):.3f} GeV, mean={all_pt_flat.mean():.3f} GeV")
if primary_fracs:
    print(f"  Primary particle fraction: {np.mean(primary_fracs):.2%}")

## 3. Tracker Hit Characterisation & Detector Geometry

In [ ]:
n_tracker_sample = min(get_num_events(tracker_table), 2000)

hit_counts = []
all_volume_ids = []
all_layer_ids = []
sample_xyz = []  # collect positions from a few events for visualisation

for i in tqdm(range(n_tracker_sample), desc="Scanning tracker hits"):
    ev = explode_list_columns(tracker_table, i)
    n_hits = len(ev['x'])
    hit_counts.append(n_hits)
    all_volume_ids.append(ev['volume_id'])
    all_layer_ids.append(ev['layer_id'])
    if i < 5:
        sample_xyz.append((ev['x'], ev['y'], ev['z']))

hit_counts = np.array(hit_counts)
all_volume_flat = np.concatenate(all_volume_ids)
all_layer_flat = np.concatenate(all_layer_ids)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Hit count distribution
axes[0, 0].hist(hit_counts, bins=60, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Tracker hits per event')
axes[0, 0].set_ylabel('Events')
axes[0, 0].set_title(f'Tracker hits per event (mean={hit_counts.mean():.1f})')

# Volume ID distribution (detector subsystems)
vol_unique, vol_counts = np.unique(all_volume_flat, return_counts=True)
axes[0, 1].bar([str(v) for v in vol_unique], vol_counts, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Volume ID')
axes[0, 1].set_ylabel('Hits')
axes[0, 1].set_title('Hits per detector volume')
axes[0, 1].tick_params(axis='x', rotation=45)

# Layer ID distribution
layer_unique, layer_counts = np.unique(all_layer_flat, return_counts=True)
axes[1, 0].bar([str(l) for l in layer_unique], layer_counts, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Layer ID')
axes[1, 0].set_ylabel('Hits')
axes[1, 0].set_title(f'Hits per layer ({len(layer_unique)} unique layers)')
axes[1, 0].tick_params(axis='x', rotation=45)

# x-y view of hits from first event
x0, y0, z0 = sample_xyz[0]
axes[1, 1].scatter(x0, y0, s=1, alpha=0.5)
axes[1, 1].set_xlabel('x [mm]')
axes[1, 1].set_ylabel('y [mm]')
axes[1, 1].set_title('Tracker hits x-y (event 0)')
axes[1, 1].set_aspect('equal')

plt.tight_layout()
plt.savefig('../plots/phase0_tracker_geometry.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDetector geometry summary:")
print(f"  Volume IDs: {sorted(vol_unique.tolist())}")
print(f"  Layer IDs: {sorted(layer_unique.tolist())}")
print(f"  Tracker hits/event: mean={hit_counts.mean():.1f}, std={hit_counts.std():.1f}")

In [ ]:
# r-z view of tracker hits (multiple events overlaid)
fig, ax = plt.subplots(figsize=(14, 5))
for x, y, z in sample_xyz[:3]:
    r = np.sqrt(x**2 + y**2)
    ax.scatter(z, r, s=0.5, alpha=0.3)
ax.set_xlabel('z [mm]')
ax.set_ylabel('r [mm]')
ax.set_title('Tracker r-z view (3 events overlaid)')
plt.tight_layout()
plt.savefig('../plots/phase0_tracker_rz.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Calorimeter Hit Characterisation

In [ ]:
n_calo_sample = min(get_num_events(calo_table), 2000)

calo_hit_counts = []
all_calo_energy = []
all_calo_detector = []
event_total_energy = []
sample_calo_xyz = []

for i in tqdm(range(n_calo_sample), desc="Scanning calo hits"):
    ev = explode_list_columns(calo_table, i)
    n_hits = len(ev['x'])
    calo_hit_counts.append(n_hits)
    all_calo_energy.append(ev['total_energy'])
    all_calo_detector.append(ev['detector'])
    event_total_energy.append(ev['total_energy'].sum())
    if i < 5:
        sample_calo_xyz.append((ev['x'], ev['y'], ev['z'], ev['total_energy']))

calo_hit_counts = np.array(calo_hit_counts)
all_calo_energy_flat = np.concatenate(all_calo_energy)
all_calo_det_flat = np.concatenate(all_calo_detector)
event_total_energy = np.array(event_total_energy)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Calo hit count distribution
axes[0, 0].hist(calo_hit_counts, bins=60, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Calorimeter hits per event')
axes[0, 0].set_ylabel('Events')
axes[0, 0].set_title(f'Calo hits/event (mean={calo_hit_counts.mean():.1f})')

# Energy spectrum
axes[0, 1].hist(all_calo_energy_flat[all_calo_energy_flat > 0],
                bins=np.logspace(-5, 2, 100), edgecolor='black', alpha=0.7)
axes[0, 1].set_xscale('log')
axes[0, 1].set_yscale('log')
axes[0, 1].set_xlabel('Cell energy [GeV]')
axes[0, 1].set_ylabel('Cells')
axes[0, 1].set_title('Calorimeter cell energy spectrum')

# Detector subsystem distribution
det_unique, det_counts = np.unique(all_calo_det_flat, return_counts=True)
axes[1, 0].bar([str(d) for d in det_unique], det_counts, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Detector code')
axes[1, 0].set_ylabel('Hits')
axes[1, 0].set_title(f'Hits per calo subsystem ({len(det_unique)} subsystems)')

# Total event energy distribution
axes[1, 1].hist(event_total_energy, bins=60, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Total calorimeter energy [GeV]')
axes[1, 1].set_ylabel('Events')
axes[1, 1].set_title(f'Total calo energy/event (mean={event_total_energy.mean():.2f} GeV)')

plt.tight_layout()
plt.savefig('../plots/phase0_calo_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Calorimeter η-φ occupancy map
x0, y0, z0, e0 = sample_calo_xyz[0]
r0 = np.sqrt(x0**2 + y0**2)
eta0 = np.arctanh(np.clip(z0 / (np.sqrt(x0**2 + y0**2 + z0**2) + 1e-10), -1+1e-7, 1-1e-7))
phi0 = np.arctan2(y0, x0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# eta-phi scatter weighted by energy
sc = axes[0].scatter(eta0, phi0, c=e0, s=5, cmap='hot', norm=LogNorm(vmin=max(e0.min(), 1e-5)))
plt.colorbar(sc, ax=axes[0], label='Energy [GeV]')
axes[0].set_xlabel('η')
axes[0].set_ylabel('φ')
axes[0].set_title('Calo hits η-φ (event 0, color=energy)')

# r-z view
sc2 = axes[1].scatter(z0, r0, c=e0, s=5, cmap='hot', norm=LogNorm(vmin=max(e0.min(), 1e-5)))
plt.colorbar(sc2, ax=axes[1], label='Energy [GeV]')
axes[1].set_xlabel('z [mm]')
axes[1].set_ylabel('r [mm]')
axes[1].set_title('Calo hits r-z (event 0)')

plt.tight_layout()
plt.savefig('../plots/phase0_calo_etaphi.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Particle → Hit Mapping

In [ ]:
# Build particle-hit map for a sample event
evt_idx = 0

p_ev = explode_list_columns(particles_table, evt_idx)
t_ev = explode_list_columns(tracker_table, evt_idx)
c_ev = explode_list_columns(calo_table, evt_idx)

hit_map = get_particle_hit_map(p_ev, t_ev, c_ev)

# Statistics
n_with_tracker = sum(1 for v in hit_map.values() if len(v['tracker_hits']) > 0)
n_with_calo = sum(1 for v in hit_map.values() if len(v['calo_hits']) > 0)
n_with_both = sum(1 for v in hit_map.values() if len(v['tracker_hits']) > 0 and len(v['calo_hits']) > 0)
n_with_neither = sum(1 for v in hit_map.values() if len(v['tracker_hits']) == 0 and len(v['calo_hits']) == 0)

print(f"Event {evt_idx}:")
print(f"  Total particles: {len(hit_map)}")
print(f"  With tracker hits: {n_with_tracker} ({n_with_tracker/len(hit_map):.1%})")
print(f"  With calo hits: {n_with_calo} ({n_with_calo/len(hit_map):.1%})")
print(f"  With both: {n_with_both} ({n_with_both/len(hit_map):.1%})")
print(f"  With neither: {n_with_neither} ({n_with_neither/len(hit_map):.1%})")

In [ ]:
# Distribution of tracker hits per particle and calo hits per particle
trk_per_particle = [len(v['tracker_hits']) for v in hit_map.values()]
calo_per_particle = [len(v['calo_hits']) for v in hit_map.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(trk_per_particle, bins=range(0, max(trk_per_particle)+2),
             edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Tracker hits per particle')
axes[0].set_ylabel('Particles')
axes[0].set_title('Tracker hits per particle')

axes[1].hist(calo_per_particle, bins=range(0, min(max(calo_per_particle)+2, 50)),
             edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Calo hits per particle')
axes[1].set_ylabel('Particles')
axes[1].set_title('Calo hits per particle')

plt.tight_layout()
plt.savefig('../plots/phase0_hits_per_particle.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Tracker hits/particle: mean={np.mean(trk_per_particle):.2f}, max={max(trk_per_particle)}")
print(f"Calo hits/particle: mean={np.mean(calo_per_particle):.2f}, max={max(calo_per_particle)}")

In [ ]:
# Tracker hits per particle vs pT (particles with hits only)
pts_with_hits = []
nhits_with_hits = []
for pid, v in hit_map.items():
    if len(v['tracker_hits']) > 0:
        px = v['kinematics'].get('px', 0)
        py = v['kinematics'].get('py', 0)
        pt = np.sqrt(px**2 + py**2)
        pts_with_hits.append(pt)
        nhits_with_hits.append(len(v['tracker_hits']))

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(pts_with_hits, nhits_with_hits, s=3, alpha=0.3)
ax.set_xlabel('pT [GeV/c]')
ax.set_ylabel('Number of tracker hits')
ax.set_title('Tracker hit count vs particle pT')
ax.set_xscale('log')
plt.tight_layout()
plt.savefig('../plots/phase0_nhits_vs_pt.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary for Model Design

Review the outputs above to answer:
- What is the typical particle multiplicity per min-bias interaction?
- What is the pT distribution? (Mostly soft, with steeply falling spectrum)
- How many tracker layers does the ODD have, and which volume IDs correspond to which sub-detectors?
- What is the typical number of tracker hits / calo hits per particle?
- What fraction of particles leave no detector signature at all?
- What are the calorimeter subsystem codes?